In [1]:
import torch
print(torch.cuda.is_available())

True


## **Training RGB yolo model**

In [3]:
from ultralytics import YOLO

# Load pretrained YOLOv8 small model
model = YOLO("yolov8s.pt")

model.train(
    data="D:/IITBHU Internship/code/DroneDatasetCombined/data.yaml",

    # Training duration
    epochs=100,              # allow learning, early stopping will cut it
    patience=10,             # early stopping (good)

    # Image & batch
    imgsz=768,               # good for small drones
    batch=16,
    device=0,
    workers=0,
    cache=False,

    # ===== LOSS WEIGHTING (VERY IMPORTANT) =====
    box=12.0,                # ↑ box loss (default ~7.5)
    cls=2.0,                 # ↑ class loss (default ~0.5)

    # ===== SMALL OBJECT–FRIENDLY AUGMENTATION =====
    scale=0.9,               # aggressive scaling (shrinks objects)
    translate=0.15,          # simulate motion
    mosaic=1.0,              # KEEP mosaic ON
    mixup=0.2,               # helps generalization
    copy_paste=0.2,          # improves recall
    degrees=0.0,             # no rotation (important for aerial)
    shear=0.0,
    perspective=0.0,

    # ===== RECALL BIAS (FN > FP) =====
    conf=0.15,               # lower confidence threshold
    iou=0.5,                  # allow more matches
)


SyntaxError: keyword argument repeated: cache (3242760128.py, line 37)

## **testing on test dataset**

In [ ]:
from ultralytics import YOLO

# 1) Load your trained model
model = YOLO(r"D:\IITBHU Internship\code\runs\detect\train10\weights\best.pt")   # adjust path if needed

# 2) Evaluate on your test dataset
results = model.val(
    data="D:/IITBHU Internship/code/DroneDatasetCombined/data.yaml",
    split="test",   # <<< ensures evaluation on test set
    imgsz=768,
    batch=16,
    device=0,
    conf=0.1,
    iou=0.5
)

# 3) Print results
print(results)


Ultralytics 8.3.235  Python-3.11.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.30.2 ms, read: 51.719.4 MB/s, size: 17.3 KB)
val: Scanning D:\IITBHU Internship\code\DroneDatasetCombined\labels\test.cache... 521 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 521/521 256.9Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 33/33 3.5it/s 9.5s0.3s
                   all        521        588      0.927      0.908      0.954      0.553
Speed: 1.8ms preprocess, 10.6ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to D:\IITBHU Internship\code\runs\detect\val2
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0

## Real-time yolo model drone detection

In [1]:
import cv2
from ultralytics import YOLO

model = YOLO(r"D:\IITBHU Internship\code\runs\detect\train11\best.pt") 

#web camera access
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Webcam not opening")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Frame not found, breaking...")
        break

    # 3) YOLO se prediction (frame BGR hi pass kar sakte ho)
    results = model(
        frame,
        conf=0.15,      # confidence threshold
        device=0,       # GPU: 0, agar GPU issue ho to "cpu"
        iou=0.5,
        verbose=False   # console me spam kam
    )

    # 4) Annotated frame (bounding boxes + labels drawn)
    annotated_frame = results[0].plot()  # numpy array (BGR)

    # 5) Show in window
    cv2.imshow("Drone/Bird/Airplane Detection", annotated_frame)

    # 6) 'q' to exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 7) Cleanup
cap.release()
cv2.destroyAllWindows()
